# Day 13: Pathway enrichment

## この章で解く現実の課題

上昇遺伝子群には、炎症応答に関係する遺伝子が偶然以上に多く含まれているのかを調べます。

差次的発現解析で得られるのは遺伝子リストです。しかし研究で知りたいのは、細胞でどんな機能や経路が動いたかです。pathway enrichmentは、遺伝子リストに特定カテゴリの遺伝子が偏って含まれるかを調べます。


## enrichmentで比較しているもの

たとえば背景に1000遺伝子あり、そのうち50遺伝子が炎症関連だとします。上昇遺伝子100個の中に炎症関連遺伝子が30個入っていたら、それは偶然にしては多いでしょうか。

この問いは、データサイエンスで言えば「母集団のカテゴリ比率に対して、抽出された集合でカテゴリが過剰に出ているか」という問題です。


In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt

def hypergeom_pmf(k, population_size, success_in_population, draws):
    # 背景population_size個のうちsuccess_in_population個がpathway遺伝子。
    # draws個を選んだとき、ちょうどk個がpathway遺伝子である確率。
    if k < 0 or k > success_in_population or k > draws:
        return 0.0
    failures = population_size - success_in_population
    if draws - k > failures:
        return 0.0
    return (
        math.comb(success_in_population, k)
        * math.comb(failures, draws - k)
        / math.comb(population_size, draws)
    )

def hypergeom_sf(k_minus_1, population_size, success_in_population, draws):
    # P(X > k_minus_1)、つまり P(X >= k_minus_1 + 1) を計算する。
    start = k_minus_1 + 1
    end = min(success_in_population, draws)
    return sum(
        hypergeom_pmf(k, population_size, success_in_population, draws)
        for k in range(start, end + 1)
    )

universe_size = 1000          # 背景遺伝子数
pathway_size = 50             # 背景のうち炎症pathwayに属する遺伝子数
selected_size = 100           # 上昇遺伝子数
overlap = 30                  # 上昇遺伝子のうち炎症pathwayに属する数

# P(X >= overlap) を計算する。
# 偶然selected_size個を選んだとき、pathway遺伝子がoverlap個以上入る確率。
p_value = hypergeom_sf(overlap - 1, universe_size, pathway_size, selected_size)
p_value


In [ ]:
expected_overlap = selected_size * pathway_size / universe_size
pd.DataFrame({
    "quantity": ["observed overlap", "expected overlap by chance", "p-value"],
    "value": [overlap, expected_overlap, p_value],
})


## 複数pathwayを調べる

現実には、炎症応答だけでなく、細胞周期、アポトーシス、代謝など多くのpathwayを同時に調べます。そのため、多重検定補正が必要です。


In [ ]:
pathway_results = pd.DataFrame({
    "pathway": ["inflammatory response", "TNF signaling", "cell cycle", "oxidative phosphorylation", "ribosome"],
    "universe_size": [1000, 1000, 1000, 1000, 1000],
    "pathway_size": [50, 40, 120, 90, 80],
    "selected_size": [100, 100, 100, 100, 100],
    "overlap": [30, 20, 8, 6, 7],
})

pathway_results["p_value"] = pathway_results.apply(
    lambda r: hypergeom_sf(r["overlap"] - 1, r["universe_size"], r["pathway_size"], r["selected_size"]),
    axis=1,
)

# Benjamini-Hochberg FDRの簡易実装。
pathway_results = pathway_results.sort_values("p_value").reset_index(drop=True)
m = len(pathway_results)
pathway_results["rank"] = pathway_results.index + 1
pathway_results["bh_fdr"] = (pathway_results["p_value"] * m / pathway_results["rank"]).clip(upper=1)
pathway_results


In [ ]:
ax = pathway_results.sort_values("bh_fdr").plot(
    x="pathway", y="overlap", kind="barh", figsize=(7, 4), legend=False
)
ax.set_xlabel("overlap with up-regulated genes")
ax.set_title("Pathways enriched among up-regulated genes")
plt.tight_layout()
plt.show()


## 読み取り

`inflammatory response` や `TNF signaling` が上位に来るなら、差次的発現遺伝子の解釈として「炎症応答が活性化した」という仮説を立てやすくなります。

ただし、enrichmentは証明ではありません。背景遺伝子集合、遺伝子IDの対応、DEGの閾値、pathway databaseの偏りに影響されます。

## エラーが出た場合

- `No module named scipy`: Colabでは通常使えます。ローカルではvenvにscipyが必要です。
- FDRが単調でない: 厳密なBH補正では累積最小化を使います。ここでは概念理解用の簡易版です。

## この章のゴール

enrichmentのp値が「背景遺伝子集合から偶然選んだ場合と比べた偏り」を表すと説明できれば合格です。
